# ReviewLens AI — Model Training
**Run this notebook on Google Colab with a T4 GPU (free)**

This notebook will:
1. Install dependencies
2. Download the Amazon Reviews dataset from Kaggle
3. Run the data pipeline
4. Fine-tune DistilBERT for sentiment analysis
5. Fine-tune T5-small for summarization
6. Package both models into a ZIP for download

**Estimated time: ~25-35 minutes on T4 GPU**

---
### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your `kaggle.json` file when asked in Cell 2

In [6]:
!pip install numpy==1.26.4

In [1]:
# ── Cell 1: Check GPU ─────────────────────────────────────────
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU')

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [2]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.33.0 \
    evaluate==0.4.2 rouge-score==0.1.2 sentencepiece==0.2.0 \
    scikit-learn==1.5.2 seaborn kaggle

In [3]:
# ── Cell 3: Upload kaggle.json ────────────────────────────────
# This will show a file upload button
# Upload your kaggle.json from: https://www.kaggle.com/settings > API > Create New Token
from google.colab import files
import os

print('Upload your kaggle.json file...')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json configured!')

Upload your kaggle.json file...


Saving kaggle.json to kaggle.json
kaggle.json configured!


In [4]:
# ── Cell 4: Download dataset ──────────────────────────────────
import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('saved_models/sentiment', exist_ok=True)
os.makedirs('saved_models/summarization', exist_ok=True)

# downloading a different dataset that's always available
!kaggle datasets download -d snap/amazon-fine-food-reviews -p data/raw/ --unzip

import glob
files_found = glob.glob('data/raw/*')
print(f'Files in data/raw/: {files_found}')

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 242M/242M [00:12<00:00, 20.8MB/s]

Files in data/raw/: ['data/raw/Reviews.csv', 'data/raw/hashes.txt', 'data/raw/database.sqlite']


In [5]:
# ── Cell 5: Data pipeline ─────────────────────────────────────
import re, html, glob
import pandas as pd
from sklearn.model_selection import train_test_split

LABEL_MAP = {1: 0, 2: 0, 3: 1, 4: 2, 5: 2}

def clean_text(text):
    if not isinstance(text, str): return ''
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.read_csv('data/raw/Reviews.csv', nrows=150000)
print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')

df = df.rename(columns={'Text': 'review_body', 'Score': 'star_rating', 'Summary': 'review_headline'})
df['review_body'] = df['review_body'].apply(clean_text)
df['review_headline'] = df['review_headline'].apply(clean_text)
df = df.dropna(subset=['review_body', 'star_rating'])
df = df[df['review_body'].str.len() > 20]
df = df.drop_duplicates(subset=['review_body'])
df['star_rating'] = pd.to_numeric(df['star_rating'], errors='coerce').astype(int)
df = df[df['star_rating'].between(1, 5)]
df['label'] = df['star_rating'].map(LABEL_MAP)

# --- NEW: Balance classes to prevent positive bias ---
min_class_size = df['label'].value_counts().min()
df = df.groupby('label', group_keys=False).apply(lambda x: x.sample(min_class_size, random_state=42)).reset_index(drop=True)
# ----------------------------------------------------

print(f'\nFinal Balanced: {len(df):,} rows')
print(df['label'].value_counts().sort_index())

cols = ['review_body', 'star_rating', 'label', 'review_headline']
train_df, temp = train_test_split(df[cols], test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp, test_size=0.50, stratify=temp['label'], random_state=42)

train_df.to_csv('data/processed/train.csv', index=False)
val_df.to_csv('data/processed/val.csv', index=False)
test_df.to_csv('data/processed/test.csv', index=False)

print(f'\nTrain: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}')
print('Data pipeline complete! ✅')

Loaded 150,000 rows
Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


/tmp/ipykernel_1911/2703583337.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('label', group_keys=False).apply(lambda x: x.sample(min_class_size, random_state=42)).reset_index(drop=True)



Final Balanced: 30,309 rows
label
0    10103
1    10103
2    10103
Name: count, dtype: int64

Train: 21,216, Val: 4,546, Test: 4,547
Data pipeline complete! ✅


In [6]:
# ── Cell 6: Train DistilBERT (Sentiment) ──────────────────────
# ~15 minutes on T4 GPU
import torch
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    DistilBertForSequenceClassification, DistilBertTokenizerFast,
    Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
)

ID2LABEL = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
LABEL2ID = {'Negative': 0, 'Neutral': 1, 'Positive': 2}

train_df = pd.read_csv('data/processed/train.csv')
val_df   = pd.read_csv('data/processed/val.csv')
test_df  = pd.read_csv('data/processed/test.csv')

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def make_ds(df):
    ds = Dataset.from_pandas(df[['review_body','label']].rename(columns={'review_body':'text'}))
    return ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=256, padding=False),
                  batched=True, remove_columns=['text'])

train_ds = make_ds(train_df)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': round(acc,4), 'f1_macro': round(f1,4),
            'precision': round(p,4), 'recall': round(r,4)}

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)

args = TrainingArguments(
    output_dir='saved_models/sentiment/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=50,
    save_total_limit=1,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Training DistilBERT...')
trainer.train()

print('\nEvaluating on test set...')
results = trainer.evaluate(test_ds)
print(f'Test results: {results}')

trainer.save_model('saved_models/sentiment')
tokenizer.save_pretrained('saved_models/sentiment')
print('Sentiment model saved!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Training DistilBERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision,Recall
1,0.572100,0.591757,0.751000,0.741300,0.748800,0.751000


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision,Recall
1,0.572100,0.591757,0.751000,0.741300,0.748800,0.751000
2,0.469700,0.541230,0.776900,0.776700,0.776900,0.776900
3,0.413100,0.560453,0.778000,0.779000,0.780400,0.778000



Evaluating on test set...


Test results: {'eval_loss': 0.5718821287155151, 'eval_accuracy': 0.7667, 'eval_f1_macro': 0.7676, 'eval_precision': 0.7689, 'eval_recall': 0.7667, 'eval_runtime': 11.4139, 'eval_samples_per_second': 398.376, 'eval_steps_per_second': 6.308, 'epoch': 3.0}
Sentiment model saved!


In [7]:
# ── Cell 7: Train FLAN-T5-base (Simple training loop) ─────────────
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import T5ForConditionalGeneration, T5TokenizerFast
from torch.optim import AdamW

TASK_PREFIX = 'summarize: '
MAX_INPUT = 256
MAX_TARGET = 64
EPOCHS = 2
BATCH_SIZE = 8
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

train_df = pd.read_csv('data/processed/train.csv')
val_df   = pd.read_csv('data/processed/val.csv')

# --- NEW: Drop reviews without headlines to prevent copy bias ---
train_df = train_df.dropna(subset=['review_headline'])
train_df = train_df[train_df['review_headline'].str.strip().str.len() > 5].reset_index(drop=True)
train_df['target'] = train_df['review_headline'].astype(str)

val_df = val_df.dropna(subset=['review_headline'])
val_df = val_df[val_df['review_headline'].str.strip().str.len() > 5].reset_index(drop=True)
val_df['target'] = val_df['review_headline'].astype(str)
# ----------------------------------------------------------------

train_df = train_df[train_df['review_body'].str.split().str.len() > 15].reset_index(drop=True)
val_df   = val_df[val_df['review_body'].str.split().str.len() > 15].reset_index(drop=True)

# --- CHANGED: Using google/flan-t5-base instead of t5-base ---
t5_tokenizer = T5TokenizerFast.from_pretrained('google/flan-t5-base')
t5_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base').to(device)

class ReviewDataset(TorchDataset):
    def __init__(self, df):
        self.inputs  = [TASK_PREFIX + t for t in df['review_body'].tolist()]
        self.targets = df['target'].tolist()
    def __len__(self): return len(self.inputs)
    def __getitem__(self, i): return self.inputs[i], self.targets[i]

def collate_fn(batch):
    src, tgt = zip(*batch)
    enc = t5_tokenizer(list(src), max_length=MAX_INPUT, truncation=True, padding=True, return_tensors='pt')
    dec = t5_tokenizer(list(tgt), max_length=MAX_TARGET, truncation=True, padding=True, return_tensors='pt')
    labels = dec['input_ids'].clone()
    labels[labels == t5_tokenizer.pad_token_id] = -100
    return enc['input_ids'], enc['attention_mask'], labels

train_loader = DataLoader(ReviewDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(ReviewDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

optimizer = AdamW(t5_model.parameters(), lr=5e-5)

# --- CHANGED: Updated print statement for FLAN-T5-base ---
print('Training FLAN-T5-base...')
for epoch in range(EPOCHS):
    t5_model.train()
    total_loss = 0
    for i, (input_ids, attention_mask, labels) in enumerate(train_loader):
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        loss = t5_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if i % 100 == 0:
            print(f'  Epoch {epoch+1} | Step {i}/{len(train_loader)} | Loss: {loss.item():.4f}')
    print(f'Epoch {epoch+1} done. Avg loss: {total_loss/len(train_loader):.4f}')

t5_model.save_pretrained('saved_models/summarization')
t5_tokenizer.save_pretrained('saved_models/summarization')
print('Summarization model saved! ✅')

Using: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Training FLAN-T5-base...
  Epoch 1 | Step 0/2549 | Loss: 3.2418
  Epoch 1 | Step 100/2549 | Loss: 2.5250
  Epoch 1 | Step 200/2549 | Loss: 2.5617
  Epoch 1 | Step 300/2549 | Loss: 2.6418
  Epoch 1 | Step 400/2549 | Loss: 3.0817
  Epoch 1 | Step 500/2549 | Loss: 3.4282
  Epoch 1 | Step 600/2549 | Loss: 3.4515
  Epoch 1 | Step 700/2549 | Loss: 3.0983
  Epoch 1 | Step 800/2549 | Loss: 2.7150
  Epoch 1 | Step 900/2549 | Loss: 3.1949
  Epoch 1 | Step 1000/2549 | Loss: 2.9770
  Epoch 1 | Step 1100/2549 | Loss: 3.1650
  Epoch 1 | Step 1200/2549 | Loss: 3.5690
  Epoch 1 | Step 1300/2549 | Loss: 3.4886
  Epoch 1 | Step 1400/2549 | Loss: 2.2980
  Epoch 1 | Step 1500/2549 | Loss: 2.5407
  Epoch 1 | Step 1600/2549 | Loss: 2.7883
  Epoch 1 | Step 1700/2549 | Loss: 2.9676
  Epoch 1 | Step 1800/2549 | Loss: 3.1741
  Epoch 1 | Step 1900/2549 | Loss: 2.9499
  Epoch 1 | Step 2000/2549 | Loss: 2.7891
  Epoch 1 | Step 2100/2549 | Loss: 2.8941
  Epoch 1 | Step 2200/2549 | Loss: 2.2736
  Epoch 1 | Step 2300

In [8]:
# ── test_local_summary.py (Local Fine-tuned Model Tester) ──
import os
import torch
from transformers import T5ForConditionalGeneration, T5TokenizerFast

# Ye aapke local project folder se model read karega
model_path = "saved_models/summarization"

print(f"Loading local fine-tuned model from: {model_path}...")
tokenizer = T5TokenizerFast.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

configs = [
    {"min_length": 6, "max_new_tokens": 64, "num_beams": 4, "length_penalty": 1.0, "no_repeat_ngram_size": 3},
    {"min_length": 10, "max_new_tokens": 64, "num_beams": 4, "length_penalty": 1.2, "no_repeat_ngram_size": 3},
    {"min_length": 12, "max_new_tokens": 64, "num_beams": 4, "length_penalty": 1.5, "no_repeat_ngram_size": 3},
    {"min_length": 12, "max_new_tokens": 64, "num_beams": 4, "length_penalty": 1.2, "repetition_penalty": 1.2, "no_repeat_ngram_size": 3}
]

test_custom_reviews = [
    "The delivery was fast and the packaging was good, but the product quality was just average. It works fine for now, although I expected better durability. Overall, it's okay but not something I would purchase again.",
    "I really wanted to like this product because the design looks beautiful, but it stopped working after only a few days. Customer support took too long to respond, and the replacement process was frustrating. I cannot recommend it.",
    "The product had a small scratch when it arrived, but everything else was excellent. The performance is amazing, and the quality is impressive. Despite the minor issue, I am happy with my purchase."
]

def test_summary_local(text, cfg):
    prefixed = "summarize: " + text
    inputs = tokenizer(prefixed, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(**inputs, **cfg, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- Tuning Parameters for Local Model ---")
for r_idx, rev in enumerate(test_custom_reviews):
    print(f"\nOriginal Review {r_idx+1}:\n{rev}")
    for c_idx, cfg in enumerate(configs):
        print(f"  Config {c_idx+1} (min={cfg['min_length']}, penalty={cfg['length_penalty']}): {test_summary_local(rev, cfg)}")

Loading local fine-tuned model from: saved_models/summarization...

--- Tuning Parameters for Local Model ---

Original Review 1:
The delivery was fast and the packaging was good, but the product quality was just average. It works fine for now, although I expected better durability. Overall, it's okay but not something I would purchase again.
  Config 1 (min=6, penalty=1.0): Good product, but average quality
  Config 2 (min=10, penalty=1.2): Good product, but not something I would buy again
  Config 3 (min=12, penalty=1.5): Good product, but not something I would buy again.
  Config 4 (min=12, penalty=1.2): Good product, but not something I would buy again.

Original Review 2:
I really wanted to like this product because the design looks beautiful, but it stopped working after only a few days. Customer support took too long to respond, and the replacement process was frustrating. I cannot recommend it.
  Config 1 (min=6, penalty=1.0): Doesn't last long
  Config 2 (min=10, penalty=1.2):

In [9]:
# ── Cell 8: Package and download models ───────────────────────
# This zips both trained models so you can download them
import shutil

print('Packaging models...')
shutil.make_archive('reviewlens_models', 'zip', '.', 'saved_models')

import os
size_mb = os.path.getsize('reviewlens_models.zip') / (1024*1024)
print(f'ZIP size: {size_mb:.1f} MB')
print('Downloading...')

from google.colab import files
files.download('reviewlens_models.zip')
print('Done! Extract the ZIP and place the saved_models/ folder in your project root.')

Packaging models...
ZIP size: 1742.3 MB
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Extract the ZIP and place the saved_models/ folder in your project root.


## After downloading

1. Extract `reviewlens_models.zip`
2. You'll see a `saved_models/` folder with `sentiment/` and `summarization/` inside
3. Copy both folders into your project:
   ```
   reviewlens-ai/
   └── saved_models/
       ├── sentiment/      ← paste here
       └── summarization/  ← paste here
   ```
4. Restart your Flask app:
   ```powershell
   python app.py
   ```
5. Header should now show **models ready** in green!
   
---
*ReviewLens AI — Fine-tuned on Amazon Electronics Reviews*